### Checks

Merge `Urban Ward_Panel` and `Urban Ward_IFS` sheets. Add 2 columns, in_IFS and in_Panel.


1. All TV_CODES should show up at least once. Either one shape for a town or village (NaN Ward code) or multiple ward shapes with codes.
2. Which TV_Codes have we been given multiple shapes for (wards)? Should match or overdeliver on their promise.

Add column that says "Delivery State": 
- "FAIL - No shape(s) given", 
- "PASS - single Town/Village shape given (expected)", 
- "FAIL - only town shape given (but Ward shapes promised)", 
- "PASS - Ward shapes given", 
- "FAIL - Ward shapes given but sampled ward missing", 
- "PASS - subdistrict given only (expected)"

3. Also check population/summed up population matches the value in our reference.
4. Subdistrict codes for 3 states should show up for those states



#### OLD

- Sheet `Urban Ward_IFS` sheet: match the `TownVillage`-`UrbanWardVillage` columns (where it's not 0) to what's in the given dataset for each state - do these sampled wards exist in the shared shapes? Add column `Delivered by MapSolve`
    - If not, was it flagged as available? If yes flag as "Promised but not delivered ward shapes"
    - If yes, flag as delivered


`UrbanWardVillage`:
- If it's 0, it's a rural village, and there should just be NaN for Ward_C but one shape for the TV_Code (same as town)
- If it's non-zero, it's a sampled ward. If MapSolve has said available we should get wards. If they've said unavailable, the `Ward_C` should be null but the TV_Code should be there.


Only expect subdistrict boundaries for our sampled subdistricts - only check for subdistrict code existence:
- meghalaya
- tripura
- uttarakhand

## Setup

In [ ]:
# set libraries to refresh
%load_ext autoreload
%autoreload 2

In [ ]:
# general
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
from tqdm.notebook import tqdm

# for plotting and coloring
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import math
import matplotlib.cm

gpd.options.io_engine = "pyogrio"

In [ ]:
from gridsample.utils import create_ids, create_gmap_links, save_shapefiles
from gridsample.mapping import create_interactive_map

In [ ]:
ROOT_DIR = Path("../")
DATA_DIR = ROOT_DIR / "data"
RAW_DATA_DIR = DATA_DIR / "01. Raw Data"
CLEANED_DATA_DIR = DATA_DIR / "02. Intermediate Outputs"
OUTPUT_DATA_DIR = DATA_DIR / "03. Final Outputs"
OUTPUT_DATA_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def generate_colormap(N):
    arr = np.arange(N)/N
    N_up = int(math.ceil(N/7)*7)
    arr.resize(N_up)
    arr = arr.reshape(7,N_up//7).T.reshape(-1)
    ret = matplotlib.cm.hsv(arr)
    n = ret[:,3].size
    a = n//2
    b = n-a
    
    # Create arrays of exactly matching sizes
    for i in range(3):
        ret[0:a,i] *= np.linspace(0.2, 1, a)
    ret[a:,3] *= np.linspace(1, 0.3, b)
    
    return ret[:N]  # Return only the requested number of colors

## 0. Load request excel

In [ ]:
excel_file = RAW_DATA_DIR / "00. Boundary Requests" / "SamplingOutput_Summary_IFS & Panel.xlsx"
ifs_df = pd.read_excel(excel_file, sheet_name="Urban Ward_IFS")
panel_df = pd.read_excel(excel_file, sheet_name="Urban Ward_Panel")

In [ ]:
ifs_df["Selected in IFS Sample"] = "Yes"

In [ ]:
merged_df = panel_df.merge(
    ifs_df,
    on=list(ifs_df.columns.intersection(panel_df.columns)),
    how="outer",
    indicator="Source Sheet",
)
merged_df["Source Sheet"] = merged_df["Source Sheet"].replace(
    {
        "both": "Both IFS and Panel",
        "left_only": "IFS",
        "right_only": "Panel",
    }
)
merged_df

In [ ]:
# rename columns for clarity
merged_df = merged_df.rename(
    columns={
        "Selected in IFS Sample": "Sampled for IFS",
        "Included in Panel": "Sampled for Panel",
    },
)

In [ ]:
# check
print("Left:", merged_df.loc[merged_df["Source Sheet"] == "IFS", "Sampled for IFS"].value_counts())
print("Right or Both", merged_df.loc[merged_df["Source Sheet"] != "IFS", "Sampled for IFS"].value_counts())

In [ ]:
# fill NaN values in "Included in Panel" column to No
merged_df["Sampled for Panel"].fillna("No", inplace=True)

# sort and reset index
merged_df = merged_df.sort_values(by=["State", "District", "Subdistrict", "TownVillage", "UrbanWardVillage"]).reset_index(drop=True)

In [ ]:
merged_df

In [ ]:
merged_df.to_csv(CLEANED_DATA_DIR / "Merged Panel and IFS Wards.csv", index=False)

In [ ]:
# get unique across district and subdistrict both
len(merged_df[["State", "District", "Subdistrict"]].drop_duplicates())

In [ ]:
subdistrict_only_state_names = ["MEGHALAYA", "TRIPURA", "UTTARAKHAND"]
merged_df.loc[merged_df["State_Name"].isin(subdistrict_only_state_names), ["State", "State_Name"]].value_counts()

In [ ]:
subdistrict_only_state_codes = [17, 16, 5]

## 1. Load all boundaries

In [ ]:
# get all filepaths that end in `gpkg` inside the RAW_DATA_DIR / 0.1. MapSolve Boundaries
boundaries_dir = RAW_DATA_DIR / "01. MapSolve Boundaries"
gpkg_files_all = list(boundaries_dir.glob("**/*.gpkg"))
gpkg_files_all = [f for f in gpkg_files_all if f.is_file()]

In [ ]:
# # drop any with the word "Sub-District" in the filename
# gpkg_files_VTW = [f for f in gpkg_files_all if "Sub-District" not in f.name]

In [ ]:
# load all shapes into one gdf
gdf_list = []
for filepath in gpkg_files_all:
    gdf_list.append(gpd.read_file(filepath))
gdf = pd.concat(gdf_list, ignore_index=True)

## 2. Checks

### Any rows with missing TV code?

In [ ]:
gdf_no_TV_code_filtered = gdf[gdf["TV_C"].isna()]
gdf_no_TV_code_filtered

In [ ]:
gdf_no_TV_code_filtered.to_csv(CLEANED_DATA_DIR / "MapSolve Data without TV Codes.csv", index=False)

### Request satisfaction check

In [ ]:
given_states_list = list(gdf["State_C"].unique())
given_states_list.append(90) # manually add 90 for Telangana / Andhra Pradesh discrepency
given_states_list

In [ ]:
merged_df.loc[merged_df["State"].isin(given_states_list), "State_Name"].unique()

In [ ]:
merged_df["State Shared by MapSolve"] = False
merged_df.loc[merged_df["State"].isin(given_states_list), "State Shared by MapSolve"] = True

In [ ]:
merged_df["State Changed"] = "No"
merged_df.loc[merged_df["State_Name"] == "TELANGANA", "State Changed"] = "Previously Andhra Pradesh"

In [ ]:
merged_df

#### Check for wards

In [ ]:
merged_df["PCA_ID"] = merged_df["TownVillage"].astype(str) + "-" + merged_df["UrbanWardVillage"].astype(str)
given_ward_ids = gdf["PCA_ID"].unique()

merged_df["Ward Boundary Given"] = False
merged_df.loc[merged_df["PCA_ID"].isin(given_ward_ids), "Ward Boundary Given"] = True

In [ ]:
len(set(merged_df["PCA_ID"]).intersection(given_ward_ids))

#### Check for TownVillage codes

In [ ]:
given_TV_ids = set(
    gdf.loc[
        gdf["Ward_C"].isna() & gdf["TV_C"].notna(),
        "TV_C",
    ].astype(int)
)

merged_df["TV Boundary Given"] = False
merged_df.loc[
    merged_df["TownVillage"].astype(int).isin(given_TV_ids),
    "TV Boundary Given",
] = True

#### Check for SubDistricts

In [ ]:
given_subdist_ids = set(
    gdf.loc[
        gdf["Ward_C"].isna() & gdf["TV_C"].isna() & gdf["SubDist_C"].notna(),
        "SubDist_C",
    ].astype(int)
)

merged_df["SubDistrict Boundary Given"] = False
merged_df.loc[
    merged_df["Subdistrict"].astype(int).isin(given_subdist_ids),
    "SubDistrict Boundary Given",
] = True

#### Fill in Delivery State

In [ ]:
# - "FAIL - No shape(s) given", 
# - "PASS - Ward shapes given", 
# - "FAIL - only town shape given (but Ward shapes promised)", 
# - "PASS - single Town/Village shape given (expected)", 
# - "PASS - subdistrict given only (expected)"

# not implemented:
# - "FAIL - Ward shapes given but sampled ward missing", 

In [ ]:
## baseline
merged_df["Delivery State"] = "BAD - No boundary(s) given"

## subdistrict
merged_df.loc[
    (merged_df["Ward Boundary Available with MapSolve"] == "No")
    & merged_df["SubDistrict Boundary Given"],
    "Delivery State",
] = "GOOD - Subdistrict boundary given as expected"
merged_df.loc[
    (merged_df["Ward Boundary Available with MapSolve"] == "Yes")
    & merged_df["SubDistrict Boundary Given"],
    "Delivery State",
] = "BAD - Subdistrict boundary given but Ward boundaries expected"

# tripura > dukli special case
merged_df.loc[
    (merged_df["State_Name"] == "TRIPURA") & (merged_df["Subd_Name"] == "Dukli"),
    "Delivery State",
] = "GOOD - Subdistrict boundary given as expected"

## town/village
merged_df.loc[
    (merged_df["Ward Boundary Available with MapSolve"] == "No")
    & merged_df["TV Boundary Given"],
    "Delivery State",
] = "GOOD - Town/Village boundary given as expected"
merged_df.loc[
    (merged_df["Ward Boundary Available with MapSolve"] == "Yes")
    & merged_df["TV Boundary Given"],
    "Delivery State",
] = "BAD - Town/Village boundary given but Ward boundaries expected"

## ward
merged_df.loc[
    (merged_df["Ward Boundary Available with MapSolve"] == "No")
    & merged_df["Ward Boundary Given"],
    "Delivery State",
] = "BETTER - Ward boundary given but only TV or Subdistrict was expected"
merged_df.loc[
    (merged_df["Ward Boundary Available with MapSolve"] == "Yes")
    & merged_df["Ward Boundary Given"],
    "Delivery State",
] = "GOOD - Ward boundary given as expected"

In [ ]:
merged_df.to_csv(CLEANED_DATA_DIR / "Merged Wards with Quality Checks.csv", index=False)

In [ ]:
merged_df[merged_df["State Shared by MapSolve"]].to_csv(
    CLEANED_DATA_DIR / "Merged Wards with Quality Checks - Shared States.csv",
    index=False,
)

#### Save per-state stats

In [ ]:
# Get counts of delivery states by state
delivery_state_counts = (
    merged_df[merged_df["State"].isin(given_states_list)]
    .groupby("State")["Delivery State"]
    .value_counts()
)

# Convert to a more readable DataFrame format
delivery_state_pivot = delivery_state_counts.unstack(fill_value=0).reset_index()

# Sort by state code
delivery_state_pivot = delivery_state_pivot.sort_values(by="State")

# Add state names for better readability
state_name_mapping = (
    merged_df[["State", "State_Name"]]
    .drop_duplicates()
    .set_index("State")["State_Name"]
)
delivery_state_pivot["State_Name"] = delivery_state_pivot["State"].map(
    state_name_mapping
)

# Reorder columns to show State_Name first
columns = ["State", "State_Name"] + [
    col for col in delivery_state_pivot.columns if col not in ["State", "State_Name"]
]
delivery_state_pivot = delivery_state_pivot[columns]

# transform the DataFrame to have the delivery states as rows and state names as columns
delivery_state_pivot.drop(columns=["State"], inplace=True)
delivery_state_pivot = delivery_state_pivot.set_index("State_Name").T.reset_index()
delivery_state_pivot

In [ ]:
for v in delivery_state_pivot["Delivery State"]:
    print(v)

In [ ]:
delivery_state_pivot.to_csv(CLEANED_DATA_DIR / "Delivery State Counts By State.csv", index=False)

#### Old

In [ ]:
unique_requested_tv_codes = set(merged_df[merged_df["State Shared by MapSolve"]]["TownVillage"])
unique_received_tv_codes = set(gdf.loc[~gdf["TV_C"].isna(), "TV_C"].astype(int))
matched_tv_codes = unique_received_tv_codes.intersection(unique_requested_tv_codes)

In [ ]:
print("Number of requested TV codes:", len(unique_requested_tv_codes))
print("Number of received TV codes:", len(unique_received_tv_codes))

print("Number of matched TV codes:", len(matched_tv_codes))
print("Number of TV codes not received:", len(unique_requested_tv_codes.difference(matched_tv_codes)))

In [ ]:
# show the received breakdown by state
received_pivot_table = (
    merged_df[merged_df["State Shared by MapSolve"]]
    .groupby(
        [
            "State",
            "State_Name",
            "District",
            "District_Name",
            "Subdistrict",
            "Subd_Name",
        ]
    )["Delivery State"]
    .value_counts()
    .unstack(fill_value=0)
).reset_index()

# received_pivot_table.rename(
#     columns={True: "TV_Codes Received", False: "TV_Codes Not Received"}, inplace=True
# )

In [ ]:
received_pivot_table

In [ ]:
pivot_table_not_received = received_pivot_table[
    received_pivot_table["BAD - No boundary(s) given"] > 0
]
pivot_table_not_received


In [ ]:
pivot_table_not_received.to_csv(
    CLEANED_DATA_DIR / "Requested TV Codes Not Received Subdistrict Breakdown.csv", index=False
)

In [ ]:
requested_tv_codes_not_received = merged_df[
    (merged_df["State Shared by MapSolve"])
    & (merged_df["Delivery State"] == "BAD - No boundary(s) given")
]
requested_tv_codes_not_received

In [ ]:
requested_tv_codes_not_received.to_csv(
    CLEANED_DATA_DIR / "Requested TV Codes Not Received.csv", index=False
)

##### Tripura - they gave us the subdistrict boundary for Dukli

In [ ]:
gdf[(gdf["State_N"] == "Tripura")]

In [ ]:
ax = gdf[(gdf["State_N"] == "Tripura")].plot()
gdf[gdf["SubDist_N"] == "Dukli"].plot(ax=ax, color="red")

### Others

In [ ]:
# boundaries_gdf = gpd.read_file(RAW_DATA_DIR / "01. MapSolve Boundaries/State_C_7/State_C_7_Sub-District.gpkg")
# boundaries_gdf = boundaries_gdf.to_crs(4326)

# boundaries_gdf.plot(
#     column="PCA_ID",
#     figsize=(6, 6),
#     cmap=ListedColormap(generate_colormap(len(boundaries_gdf))),
#     edgecolor="black",
# )
# # create_interactive_map(boundaries_gdf, point_id_col="PCA_ID")


In [ ]:
# boundaries_gdf.plot(
#     column="PCA_ID",
#     figsize=(6, 6),
#     cmap=ListedColormap(generate_colormap(len(boundaries_gdf))),
#     edgecolor="black",
# )
# create_interactive_map(boundaries_gdf, point_id_col="PCA_ID")